# SVM (RBF + Platt tikimybės)

**Uždavinys:** $\hat p(y=1\mid x)$.

SVM randa netiesinę ribą (RBF branduolys). Natūraliai duoda ne tikimybę, o atstumą iki ribos — todėl naudojame **Platt kalibraciją** (`probability=True` sklearn viduje fitina logistinę kreivę ant decision function).

RBF SVM yra $O(n^2)$: visam 45 211 rinkiniui Colab gali užtrukti per ilgai. Todėl mokome ant **fiksuotos 6 000 eilučių** train imties (ta pati `SEED=42` — atkuriama). Tai dokumento 4.4 punkto „gali reikėti mažesnės imties“ realizacija.

Notebook savarankiškas: įkelkite `bank-full.csv`.

## 0. Bibliotekos

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_curve
from sklearn.calibration import calibration_curve

from sklearn.svm import SVC
from IPython.display import display


## 1. Bendros taisyklės ir sėkla

In [ ]:
# Fiksuota sėkla visur — paleidus du kartus rezultatai turi sutapti.
SEED = 42
C_CALL = 1.0      # sąlyginis vieno skambučio kaštas
V_SUCCESS = 10.0  # sąlyginė sėkmingo indėlio vertė (parametrai derinami su banku)
K_FRACTION = 0.10 # precision@k: k = 10 % test imties (ribotas operatorių biudžetas)

import os, random
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

DATA_PATH = "bank-full.csv"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CAT_COLS = ["job","marital","education","default","housing","loan","contact","month","poutcome"]
NUM_BASE = ["age","balance","day","campaign","pdays","previous","never_contacted"]

def load_bank(path=None):
    path = path or DATA_PATH
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Nerastas {path}. Colab: kairėje Files (aplankas) → Upload → įkelkite bank-full.csv "
            "į tą pačią sesiją kaip šis notebook."
        )
    df = pd.read_csv(path, sep=";")
    print(f"Nuskaityta: {df.shape[0]} eil. × {df.shape[1]} stulp.")
    return df

def month_change_indices(df):
    m = df["month"].to_numpy()
    ch = [0]
    for i in range(1, len(m)):
        if m[i] != m[i-1]:
            ch.append(i)
    ch.append(len(df))
    return ch

def chronological_split(df):
    """~70/15/15 pagal eilutės tvarką, ribos prie mėnesio virsmo."""
    n = len(df)
    ch = month_change_indices(df)
    def nearest(t):
        return min(ch, key=lambda c: abs(c-t))
    train_end = nearest(int(0.70 * n))
    val_end = nearest(int(0.85 * n))
    if train_end >= val_end:
        val_end = min(n, train_end + int(0.15 * n))
    train, val, test = df.iloc[:train_end].copy(), df.iloc[train_end:val_end].copy(), df.iloc[val_end:].copy()
    print(f"Skaidymas train/val/test: {len(train)}/{len(val)}/{len(test)}  ribos={train_end},{val_end}")
    print(f"  train month nuo {train['month'].iloc[0]} iki {train['month'].iloc[-1]}")
    print(f"  val   month nuo {val['month'].iloc[0]} iki {val['month'].iloc[-1]}")
    print(f"  test  month nuo {test['month'].iloc[0]} iki {test['month'].iloc[-1]}")
    return train, val, test

def temporal_shift_split(df):
    """
    Tas pats mokymas (pirmos ~70 %). Du testai:
    arti = vidurys (~15 %), toli = pabaiga (~15 %).
    """
    n = len(df)
    ch = month_change_indices(df)
    def nearest(t):
        return min(ch, key=lambda c: abs(c-t))
    train_end = nearest(int(0.70 * n))
    val_end = nearest(int(0.85 * n))
    if train_end >= val_end:
        val_end = min(n, train_end + int(0.15 * n))
    train = df.iloc[:train_end].copy()
    near = df.iloc[train_end:val_end].copy()
    far = df.iloc[val_end:].copy()
    print(
        f"Laiko poslinkis: mokymas n={len(train)}; "
        f"arti (vidurys) n={len(near)}; toli (pabaiga) n={len(far)}"
    )
    return train, near, far

def add_features(df):
    out = df.copy()
    # pdays=-1: klientas niekada nekontaktuotas anksčiau → atskiras požymis,
    # o -1 pakeičiame 0, kad tai nebūtų „neigiama trukmė“.
    out["never_contacted"] = (out["pdays"] == -1).astype(int)
    out.loc[out["pdays"] == -1, "pdays"] = 0
    out["y_bin"] = (out["y"].astype(str).str.lower() == "yes").astype(int)
    return out

def cap_previous(train, *others):
    cap = float(train["previous"].quantile(0.99))
    print(f"previous apkirpimas ties train 99-uoju procentiliu = {cap:.2f} (max buvo {train['previous'].max()})")
    out = []
    for p in (train,) + others:
        q = p.copy()
        q["previous"] = q["previous"].clip(upper=cap)
        out.append(q)
    return tuple(out)

def one_hot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_preprocessor(include_duration):
    num = NUM_BASE + (["duration"] if include_duration else [])
    return ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", one_hot(), CAT_COLS),
    ], remainder="drop")

def Xy(df, include_duration):
    cols = CAT_COLS + NUM_BASE + (["duration"] if include_duration else [])
    return df[cols].copy(), df["y_bin"].to_numpy()

def precision_at_k(y, p, k):
    k = int(min(k, len(y)))
    order = np.argsort(-np.asarray(p), kind="mergesort")
    return float(np.asarray(y)[order][:k].mean())

def contact_cost(y, yhat, c_call=C_CALL, v_success=V_SUCCESS):
    y = np.asarray(y).astype(int); yhat = np.asarray(yhat).astype(int)
    tp = int(((yhat==1)&(y==1)).sum()); fp = int(((yhat==1)&(y==0)).sum())
    return float(c_call*(tp+fp) - v_success*tp)

def best_threshold(y_val, p_val):
    cands = np.unique(np.quantile(p_val, np.linspace(0.05, 0.95, 19)))
    best_t, best_c = 0.5, np.inf
    for t in cands:
        c = contact_cost(y_val, (p_val >= t).astype(int))
        if c < best_c:
            best_c, best_t = c, float(t)
    return best_t

def metrics_dict(y, p, thr, k=None):
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float)
    if k is None:
        k = max(1, int(K_FRACTION * len(y)))
    yhat = (p >= thr).astype(int)
    return {
        "PR-AUC": float(average_precision_score(y, p)),
        "precision@k": precision_at_k(y, p, k),
        "k": int(k),
        "Brier": float(brier_score_loss(y, p)),
        "kaštai": contact_cost(y, yhat),
        "slenkstis": float(thr),
        "n": int(len(y)),
        "positives": float(y.mean()),
    }

def plot_pr(curves, title, fname):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    plt.figure(figsize=(7,5))
    for name,(y,p) in curves.items():
        pr, rc, _ = precision_recall_curve(y, p)
        ap = average_precision_score(y, p)
        plt.plot(rc, pr, label=f"{name} (PR-AUC={ap:.3f})")
    plt.xlabel("Atgaminimas (Recall)"); plt.ylabel("Tikslumas (Precision)")
    plt.title(title); plt.legend(loc="lower left"); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=140); plt.show()

def plot_cal(curves, title, fname):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    plt.figure(figsize=(7,5))
    for name,(y,p) in curves.items():
        frac, mp = calibration_curve(y, p, n_bins=10, strategy="quantile")
        plt.plot(mp, frac, marker="o", label=f"{name} (Brier={brier_score_loss(y,p):.3f})")
    plt.plot([0,1],[0,1],"k--", label="ideali kalibracija")
    plt.xlabel("Vidutinė p̂"); plt.ylabel("Stebėta teigiamų dalis")
    plt.title(title); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=140); plt.show()

def save_preds(name, y, p):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    path = os.path.join(OUTPUT_DIR, f"preds_{name}.csv")
    pd.DataFrame({"y": y, "p": p}).to_csv(path, index=False)
    print("Išsaugota", path)

def load_other_preds(exclude):
    found = {}
    if not os.path.isdir(OUTPUT_DIR):
        return found
    for fn in sorted(os.listdir(OUTPUT_DIR)):
        if fn.startswith("preds_") and fn.endswith(".csv"):
            key = fn[len("preds_"):-4]
            if key == exclude:
                continue
            tab = pd.read_csv(os.path.join(OUTPUT_DIR, fn))
            if "y" in tab.columns and "p" in tab.columns:
                found[key] = (tab["y"].to_numpy(), tab["p"].to_numpy())
    return found

def show_probability_examples(te_df, y_true, p_hat, thr, n=10, model_name="model"):
    """
    Parodo, kaip p̂ naudojama praktiškai: operatorius skambina nuo didžiausios
    tikimybės. 10 test klientų + histograma visai test imčiai.
    „Teisus/klydo“ lyginama su kaštais parinktu slenksčiu thr (ne su 0,5),
    nes prie 11,7 % teigiamos klasės p̂ dažnai būna mažesnė nei 0,5.
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    tab = te_df.copy()
    tab["p_hat"] = np.asarray(p_hat, dtype=float)
    tab["y_tikras"] = np.asarray(y_true).astype(int)
    tab["eilutes_nr"] = tab.index.astype(int)
    top = tab.sort_values("p_hat", ascending=False).head(n)
    cols = [
        c for c in [
            "eilutes_nr", "age", "job", "marital", "education", "balance",
            "housing", "loan", "contact", "month", "campaign", "poutcome",
            "never_contacted", "p_hat", "y_tikras",
        ]
        if c in top.columns
    ]
    print("10 test klientų, surikiuotų mažėjančia p̂ (kaip skambučių sąrašas):")
    display(top[cols].reset_index(drop=True))

    print("\nTas pats paprastais sakiniais:")
    for _, row in top.iterrows():
        p = float(row["p_hat"])
        tikras = "taip" if int(row["y_tikras"]) == 1 else "ne"
        pred_taip = p >= thr
        actual_taip = int(row["y_tikras"]) == 1
        verdiktas = "modelis teisus" if pred_taip == actual_taip else "modelis klydo"
        print(
            f"Klientas Nr. {int(row['eilutes_nr'])}: p̂={p:.2f} → {100*p:.0f}% "
            f"tikimybė sutikti, tikras atsakymas: {tikras} ({verdiktas})."
        )

    plt.figure(figsize=(7, 4))
    plt.hist(np.asarray(p_hat), bins=20, range=(0, 1), edgecolor="black", color="steelblue")
    plt.xlabel("p̂ (tikimybė, kad sutiks)")
    plt.ylabel("Kiek klientų")
    plt.title("Kaip pasiskirsto visos test imties tikimybės")
    plt.xlim(0, 1)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    fname = os.path.join(OUTPUT_DIR, f"prob_hist_{model_name}.png")
    plt.savefig(fname, dpi=140)
    plt.show()
    print("Histograma išsaugota:", fname)


def prepare_parts(include_duration=False):
    raw = load_bank()
    print("y=yes dalis visame rinkinyje:", (raw["y"].str.lower()=="yes").mean())
    print("pdays=-1 dalis:", (raw["pdays"]==-1).mean(), "(dokumente ~81,7 %)")
    print("duration=0 eilučių:", (raw["duration"]==0).sum(), "(visos turėtų būti y=no — nutekėjimo požymis)")
    ch = month_change_indices(raw)
    print("Mėnesio periodų:", len(ch)-1)
    df = add_features(raw)
    tr, va, te = chronological_split(df)
    tr, va, te = cap_previous(tr, va, te)
    print(
        "y=1 dalis train/val/test:",
        round(tr["y_bin"].mean(), 4),
        round(va["y_bin"].mean(), 4),
        round(te["y_bin"].mean(), 4),
        "← vėlesnėse kampanijose sutarčių daugiau; todėl chronologija būtina.",
    )
    return tr, va, te


## 2. Duomenys ir chronologinis skaidymas

In [ ]:
tr, va, te = prepare_parts(include_duration=False)


## 3. Paruošimas

SVM jautrus masteliui, todėl skaičius standartizuojame, kategorijas — one-hot. `unknown` lieka atskiri stulpeliai.

In [ ]:
MODEL_NAME = "svm"
include_duration = False
prep = make_preprocessor(include_duration)
X_tr_full = prep.fit_transform(Xy(tr, include_duration)[0])
X_va = prep.transform(Xy(va, include_duration)[0])
X_te = prep.transform(Xy(te, include_duration)[0])
y_tr_full = tr["y_bin"].to_numpy()
y_va = va["y_bin"].to_numpy()
y_te = te["y_bin"].to_numpy()

SVM_N = 6000
rng = np.random.RandomState(SEED)
# Stratifikuota imtis: išlaikome train klasės dalį, kad reta klasė neišnyktų.
pos = np.where(y_tr_full == 1)[0]
neg = np.where(y_tr_full == 0)[0]
n_pos = int(round(SVM_N * y_tr_full.mean()))
n_pos = min(max(n_pos, 1), len(pos), SVM_N - 1)
n_neg = min(SVM_N - n_pos, len(neg))
idx = np.concatenate([rng.choice(pos, n_pos, replace=False), rng.choice(neg, n_neg, replace=False)])
rng.shuffle(idx)
X_tr, y_tr = X_tr_full[idx], y_tr_full[idx]
print(f"SVM train imtis: {len(y_tr)} (pos={y_tr.mean():.3f}), viso train buvo {len(y_tr_full)}")


## 4. Mokymas

`class_weight='balanced'` — retesnei klasei didesnis svoris.
`probability=True` — Platt metodas, kad gautume $\hat p\in[0,1]$, ne tik klasės etiketę.

In [ ]:
model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight="balanced",
    probability=True,   # Platt: decision_function → tikimybė
    random_state=SEED,
)
model.fit(X_tr, y_tr)
p_va = model.predict_proba(X_va)[:, 1]
p_te = model.predict_proba(X_te)[:, 1]
thr = best_threshold(y_va, p_va)
print("Slenkstis:", round(thr, 4))


## 4.1 Tikimybės paprastai

SVM po Platt kalibracijos duoda tikimybę, ne tik „taip/ne“. Lentelė — 10 test klientų nuo labiausiai iki mažiausiai tikėtino sutikti. Histograma — visos test p̂, kad matytųsi, ar tikimybės išsibarsto, ar sulimpa ties viduriu.

In [ ]:
show_probability_examples(te, y_te, p_te, thr, n=10, model_name=MODEL_NAME)


## 5. Metrikos

In [ ]:
m = metrics_dict(y_te, p_te, thr)
display(pd.DataFrame([m], index=[MODEL_NAME]))
save_preds(MODEL_NAME, y_te, p_te)
plot_pr({MODEL_NAME: (y_te, p_te)}, "PR kreivė — SVM", "pr_svm.png")
plot_cal({MODEL_NAME: (y_te, p_te)}, "Kalibracija — SVM", "calibration_svm.png")


## 6. Abliacija ir laiko poslinkis

Abliacijoje ir poslinkyje vėl imame 6000 eilučių su ta pačia sėkla, kad būtų atkuriama ir tilptų į Colab laiką.

Laiko poslinkis: tas pats SVM (mokytas ant pirmų ~70 %) lyginamas ant vidurio vs pabaigos — ne tas pats testas dukart.

In [ ]:
def fit_svm(X, y, seed=SEED, n=SVM_N):
    rng = np.random.RandomState(seed)
    pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
    n_pos = min(max(int(round(n * y.mean())), 1), len(pos), n-1)
    n_neg = min(n - n_pos, len(neg))
    idx = np.concatenate([rng.choice(pos, n_pos, replace=False), rng.choice(neg, n_neg, replace=False)])
    clf = SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", probability=True, random_state=seed)
    clf.fit(X[idx], y[idx])
    return clf

print("=== ABLIACIJA duration ===")
prep_d = make_preprocessor(True)
Xtr_d = prep_d.fit_transform(Xy(tr, True)[0])
Xte_d = prep_d.transform(Xy(te, True)[0])
ytr_d = tr["y_bin"].to_numpy()
svm_d = fit_svm(Xtr_d, ytr_d)
p_te_d = svm_d.predict_proba(Xte_d)[:, 1]
print(f"PR-AUC be duration = {average_precision_score(y_te, p_te):.4f}")
print(f"PR-AUC su duration = {average_precision_score(y_te, p_te_d):.4f}")
plot_pr({"be duration": (y_te, p_te), "su duration": (y_te, p_te_d)}, "Abliacija duration — SVM", "pr_ablation_svm.png")

print("=== LAIKO POSLINKIS ===")
print("Tas pats SVM (pirmos ~70 %). Arti = vidurys, toli = pabaiga. Slenkstis abiem 0,5.")
print(f"y=1 dažnis arti = {y_va.mean():.4f}  ({int(y_va.sum())}/{len(y_va)})")
print(f"y=1 dažnis toli = {y_te.mean():.4f}  ({int(y_te.sum())}/{len(y_te)})")
m_near = metrics_dict(y_va, p_va, 0.5)
m_far = metrics_dict(y_te, p_te, 0.5)
cmp = pd.DataFrame([m_near, m_far], index=["arti laike (vidurys)", "toli laike (pabaiga)"])
display(cmp[["n", "positives", "PR-AUC", "precision@k", "Brier", "kaštai"]])
print(f"PR-AUC (arti − toli) = {m_near['PR-AUC'] - m_far['PR-AUC']:.4f}")
print("Jei toli PR-AUC prastesnis — modelis jautrus sąlygų kaitai laike.")
plot_pr(
    {"arti (vidurys)": (y_va, p_va), "toli (pabaiga)": (y_te, p_te)},
    "Laiko poslinkis — SVM",
    "pr_timeshift_svm.png",
)


## 7. Palyginimas su kitais modeliais

In [ ]:
# Palyginimas su kitais modeliais, jei jų notebook'ai jau paleisti (outputs/preds_*.csv)
others = load_other_preds(MODEL_NAME)
all_curves = {MODEL_NAME: (y_te, p_te), **others}
if len(all_curves) > 1:
    print("Rasti kiti modeliai:", list(others.keys()))
    rows = []
    for name,(yy,pp) in all_curves.items():
        # slenkstis 0.5 palyginimui tarp failų; tikrosios metrikos — kiekvieno notebook viduje
        rows.append({"modelis": name, **metrics_dict(yy, pp, 0.5)})
    display(pd.DataFrame(rows).set_index("modelis"))
    plot_pr(all_curves, "Precision–Recall: visi rasti modeliai", "pr_all_models.png")
    plot_cal(all_curves, "Kalibracija: visi rasti modeliai", "calibration_all_models.png")
else:
    print("Kitų modelių preds_*.csv nėra. Paleiskite kitus 3 notebook'us tame pačiame aplanke, tada perleiskite šią celę — atsiras bendri grafikai.")


Paleidus šią celę, į jūsų kompiuterio Atsisiuntimų (Downloads) aplanką atsisiųs ZIP failas su visais šio modelio rezultatais (preds_*.csv ir grafikais).

In [ ]:
import shutil
from google.colab import files
shutil.make_archive(f"{MODEL_NAME}_outputs", "zip", "outputs")
files.download(f"{MODEL_NAME}_outputs.zip")
